# Comment Category Prediction

## Submission By:

- **Name:** Varun Agnihotri
- **Roll Number:** `24f2004142`

## Import all the required libraries

In [ ]:
import gc
import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from lightgbm import LGBMClassifier
from scipy.sparse import csr_matrix, hstack, issparse
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MaxAbsScaler, OneHotEncoder
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

## Config

In [ ]:
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IS_KAGGLE:
    DATA_DIR = Path("/kaggle/input/comment-category-prediction-challenge")
else:
    DATA_DIR = Path("datasets/")

# Ensemble top K models
TOP_K = 3

# Random State for reproducibility
RANDOM_STATE = 42

# Controls HPT
PERFORM_TUNING = False

## Data loading

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_sub = pd.read_csv(
    DATA_DIR / ("Sample.csv" if IS_KAGGLE else "sample_submission.csv")
)

In [ ]:
for _name, _df in [
    ("train", train),
    ("test", test),
    ("sample_sub", sample_sub),
]:
    print(f" {_name} ".center(50, "="))
    print("\nDataset Info:")
    print(_df.info())
    print("\nDataset Description:")
    print(_df.describe(include="all"))
    # print("\nTable Head (10 rows):")
    # print(_df.head(10))
    print("\n\n")

## Helper Functions

In [ ]:
def header(msg: str) -> None:
    """
    Print a formatted header message.
    :param msg: Message to be printed as header.
    :return:
    """
    print("\n" + "=" * 50)
    print(msg.center(50))
    print("=" * 50)

In [ ]:
def clean_text(text: str) -> str:
    """
    Clean the input text by removing URLs, special characters, and extra spaces.
    :param text: Input text string.
    :return: Cleaned text string.
    """
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
def calc_sparsity(matrix) -> float:
    """
    Calculate the sparsity of a given matrix.
    :param matrix: Input matrix (numpy array or sparse matrix).
    :return: Sparsity in percentage.
    """
    total_elements = matrix.shape[0] * matrix.shape[1]
    non_zero_elements = matrix.nnz if issparse(matrix) else np.count_nonzero(matrix)
    sparsity = 1.0 - (non_zero_elements / total_elements)
    return sparsity * 100

## Exploratory Data Analysis

In [ ]:
train.shape, test.shape, sample_sub.shape

In [ ]:
train.sample(10)

In [ ]:
# Label distribution
print(train["label"].value_counts())
print(f"\nProportions:\n{train['label'].value_counts(normalize=True)}")

In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(data=train, x="label")
plt.title("Label Distribution")
plt.show()

In [ ]:
train.describe().T

In [ ]:
corr = train.corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix")
plt.show()

In [ ]:
train["comment_length"] = train["comment"].str.len()
train["word_count"] = train["comment"].str.split().str.len()

header("TEXT STATISTICS BY LABEL")
print(train.groupby("label")[["comment_length", "word_count"]].mean())

In [ ]:
cat_features = train.select_dtypes(include=("object" if IS_KAGGLE else "str")).columns
cat_features = [col for col in cat_features if col not in ["comment", "created_date"]]

header("Categorical Features")
for col in cat_features:
    print(f"\n{col} Value counts:")
    print(train[col].value_counts())
    print(f"\n{col} Proportions:")
    print(train[col].value_counts(normalize=True))
    print("\n" + "-" * 50)

## Feature Engineering

In [ ]:
def create_features(df):
    df = df.copy()

    # Text features
    df["comment_length"] = df["comment"].str.len()
    df["word_count"] = df["comment"].str.split().str.len()
    df["avg_word_length"] = df["comment"].apply(
        lambda x: np.mean([len(w) for w in str(x).split()]) if str(x).split() else 0
    )
    df["unique_word_count"] = df["comment"].apply(
        lambda x: len(set(str(x).lower().split()))
    )
    df["unique_word_ratio"] = df["unique_word_count"] / (df["word_count"] + 1)

    # Special characters and punctuation
    df["exclamation_count"] = df["comment"].str.count("!")
    df["question_count"] = df["comment"].str.count(r"\?")
    df["capital_ratio"] = df["comment"].apply(
        lambda x: sum(1 for c in str(x) if c.isupper()) / max(len(str(x)), 1)
    )
    df["digit_count"] = df["comment"].str.count(r"\d")
    df["special_char_count"] = df["comment"].str.count(r"[^a-zA-Z0-9\s]")
    df["uppercase_word_count"] = df["comment"].apply(
        lambda x: sum(1 for w in str(x).split() if w.isupper() and len(w) > 1)
    )
    df["punctuation_ratio"] = df["special_char_count"] / (df["comment_length"] + 1)

    # Engagement features
    df["total_votes"] = df["upvote"] + df["downvote"]
    df["vote_ratio"] = np.where(
        df["total_votes"] > 0, df["upvote"] / df["total_votes"], 0.5
    )
    df["vote_diff"] = df["upvote"] - df["downvote"]
    df["log_upvote"] = np.log1p(df["upvote"])
    df["log_downvote"] = np.log1p(df["downvote"])
    df["vote_product"] = np.log1p(df["upvote"] * df["downvote"])
    df["controversy_score"] = np.where(
        df["total_votes"] > 0,
        1 - np.abs(df["upvote"] - df["downvote"]) / df["total_votes"],
        0,
    )

    # Temporal features
    df["created_date"] = pd.to_datetime(df["created_date"])
    df["hour"] = df["created_date"].dt.hour
    df["day_of_week"] = df["created_date"].dt.dayofweek
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
    df["is_night"] = ((df["hour"] >= 22) | (df["hour"] <= 6)).astype(int)
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

    # Emoticon features
    df["total_emoticons"] = df["emoticon_1"] + df["emoticon_2"] + df["emoticon_3"]
    df["has_emoticon"] = (df["total_emoticons"] > 0).astype(int)

    # Convert boolean features to integers
    bool_features = df.select_dtypes(include="bool").columns
    df[bool_features] = df[bool_features].astype(int)

    # Indicator combinations
    df["indicator_sum"] = (
        (df["race"] != "none").astype(int)
        + (df["religion"] != "none").astype(int)
        + (df["gender"] != "none").astype(int)
        + df["disability"]
    )
    df["if_sum"] = df["if_1"] + df["if_2"]
    df["if_product"] = df["if_1"] * df["if_2"]

    # Post-level aggregations
    post_stats = (
        df.groupby("post_id")
        .agg(
            post_avg_upvote=("upvote", "mean"),
            post_count=("upvote", "count"),
            post_avg_downvote=("downvote", "mean"),
            post_std_upvote=("upvote", "std"),
            post_std_downvote=("downvote", "std"),
        )
        .reset_index()
    )
    post_stats = post_stats.fillna(0)

    df = df.merge(post_stats, on="post_id", how="left")
    df["upvote_vs_post"] = df["upvote"] - df["post_avg_upvote"]
    df["downvote_vs_post"] = df["downvote"] - df["post_avg_downvote"]
    df["upvote_zscore"] = df["upvote_vs_post"] / (df["post_std_upvote"] + 1e-8)
    df["downvote_zscore"] = df["downvote_vs_post"] / (df["post_std_downvote"] + 1e-8)

    # Text x engagement interactions
    df["length_x_downvote"] = df["comment_length"] * df["log_downvote"]
    df["words_x_downvote"] = df["word_count"] * df["log_downvote"]

    # Drop noisy features
    df.drop(columns=["created_date", "post_id"], inplace=True)

    return df

In [ ]:
train = create_features(train)
test = create_features(test)

In [ ]:
cat_features = train.select_dtypes(
    include=("object" if IS_KAGGLE else "str")
).columns.tolist()
cat_features.remove("comment")
cat_features

## Preprocessing

In [ ]:
cat_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="constant", fill_value="none")),
        ("onehot", OneHotEncoder(sparse_output=False, handle_unknown="ignore")),
    ]
)

transformer = ColumnTransformer(
    [("cat_process", cat_pipeline, cat_features)],
    remainder="passthrough",
    verbose_feature_names_out=False,
).set_output(transform="pandas")

# Separate label before transformation (test doesn't have label column)
y_labels = train["label"].copy()
train.drop(columns=["label"], inplace=True)

train = transformer.fit_transform(train)
test = transformer.transform(test)

# Add label back to train
train["label"] = y_labels.values

In [ ]:
header("Post-Preprocessing Dataset Info")
print(f"Total Features: {len(train.columns)}")
print(f"Output Columns: {train.columns.tolist()}")

## Model Building

In [ ]:
train["comment_clean"] = train["comment"].apply(clean_text)
test["comment_clean"] = test["comment"].apply(clean_text)

In [ ]:
# Word-level TF-IDF (Higher max_features + trigrams)
tfidf_word = TfidfVectorizer(
    max_features=4500,
    ngram_range=(1, 3),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    dtype=np.float32,
    stop_words="english",
)

# Character-level TF-IDF (Captures subword patterns/misspellings)
tfidf_char = TfidfVectorizer(
    max_features=4500,
    ngram_range=(2, 5),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    analyzer="char_wb",
    dtype=np.float32,
)

In [ ]:
tfidf_word_train = tfidf_word.fit_transform(train["comment_clean"])
tfidf_word_test = tfidf_word.transform(test["comment_clean"])

tfidf_char_train = tfidf_char.fit_transform(train["comment_clean"])
tfidf_char_test = tfidf_char.transform(test["comment_clean"])

print(f"Word TF-IDF: {tfidf_word_train.shape}")
print(f"Char TF-IDF: {tfidf_char_train.shape}")
gc.collect()

In [ ]:
feature_cols = train.select_dtypes(include="number").columns.tolist()

# remove label from features
feature_cols.remove("label")

In [ ]:
X_tab = train[feature_cols].fillna(0)
X_tab_test = test[feature_cols].fillna(0)

In [ ]:
scaler = MaxAbsScaler()
X_tab_scaled = scaler.fit_transform(X_tab)
X_tab_test_scaled = scaler.transform(X_tab_test)

In [ ]:
# Combine all features: word TF-IDF + char TF-IDF + numeric
X = hstack([tfidf_word_train, tfidf_char_train, csr_matrix(X_tab_scaled)]).tocsr()
X_sub = hstack(
    [tfidf_word_test, tfidf_char_test, csr_matrix(X_tab_test_scaled)]
).tocsr()
y = train["label"].values

print(f"Combined training shape: {X.shape}")
print(f"Combined test shape: {X_sub.shape}")
gc.collect()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

In [ ]:
header("Sparse Matrix Stats")

print("Train Matrix:")
print(f"\tShape: {X.shape}")
print(f"\tSparsity: {calc_sparsity(X):.2f}%")
print(f"\tSparse size: {X.data.nbytes / 1e6:.2f} MB")

print("\nSubmission Matrix:")
print(f"\tShape: {X_sub.shape}")
print(f"\tSparsity: {calc_sparsity(X_sub):.2f}%")
print(f"\tSparse size: {X_sub.data.nbytes / 1e6:.2f} MB")

# Cleanup
gc.collect()

In [ ]:
models = {
    # Gradient Boosting Models
    "XGBoost": XGBClassifier(
        n_estimators=1200,
        max_depth=10,
        learning_rate=0.02,
        subsample=0.8,
        colsample_bytree=0.6,
        min_child_weight=3,
        random_state=RANDOM_STATE,
        objective="multi:softprob",
        num_class=4,
        eval_metric="mlogloss",
        tree_method="hist",
        n_jobs=-1,
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=1200,
        max_depth=10,
        num_leaves=90,
        learning_rate=0.02,
        subsample=0.8,
        colsample_bytree=0.6,
        min_child_samples=20,
        random_state=RANDOM_STATE,
        verbose=-1,
        class_weight="balanced",
        n_jobs=-1,
    ),
    # # Tree-Based Ensemble Models
    # "RandomForest": RandomForestClassifier(
    #     n_estimators=300,
    #     max_depth=10,
    #     random_state=RANDOM_STATE,
    #     class_weight="balanced",
    #     n_jobs=-1,
    # ),
    # "ExtraTrees": ExtraTreesClassifier(
    #     n_estimators=300,
    #     max_depth=10,
    #     random_state=RANDOM_STATE,
    #     class_weight="balanced",
    #     n_jobs=-1,
    # ),
    # # Linear Models
    # "LogisticRegression": LogisticRegression(
    #     max_iter=3000,
    #     random_state=RANDOM_STATE,
    #     class_weight="balanced",
    #     solver="saga",
    #     n_jobs=-1,
    # ),
    # Neural Network
    "MLP": MLPClassifier(
        hidden_layer_sizes=(128, 64, 32),
        max_iter=500,
        random_state=RANDOM_STATE,
        early_stopping=True,
        validation_fraction=0.1,
    ),
}

In [ ]:
scores = {}

for model_name, model in models.items():
    header(f"Training {model_name}")
    if model_name == "XGBoost":
        sample_weights = compute_sample_weight("balanced", y_train)
        model.fit(X_train, y_train, sample_weight=sample_weights)
    else:
        model.fit(X_train, y_train)

    val_preds = model.predict(X_test)

    score = f1_score(y_test, val_preds, average="macro")
    scores[model_name] = {
        "score": score,
        "model": model,
    }
    print(f"\t{model_name} Macro F1 Score: {score:.4f}")

In [ ]:
# sort scores
scores = list(sorted(scores.items(), key=lambda x: x[1]["score"], reverse=True))

## Hyperparameter Tuning

In [ ]:
params_dict = {
    # Gradient Boosting Models
    "XGBoost": {
        "n_estimators": [200, 300, 500],
        "max_depth": [4, 6, 8],
        "learning_rate": [0.01, 0.05, 0.1],
        "min_child_weight": [1, 5, 10],
        "subsample": [0.7, 0.8, 0.9],
        "colsample_bytree": [0.7, 0.8, 0.9],
    },
    "LightGBM": {
        "n_estimators": [200, 300, 500],
        "num_leaves": [31, 63, 127],
        "learning_rate": [0.01, 0.05, 0.1],
        "min_child_samples": [10, 20, 50],
        "subsample": [0.7, 0.8, 0.9],
        "colsample_bytree": [0.7, 0.8, 0.9],
        "class_weight": ["balanced", None],
    },
    "HistGradientBoosting": {
        "max_iter": [200, 300, 500],
        "max_depth": [5, 7, 10, None],
        "learning_rate": [0.01, 0.05, 0.1],
        "min_samples_leaf": [10, 20, 50],
        "l2_regularization": [0, 0.1, 1],
        "class_weight": ["balanced", None],
    },
    # Tree-Based Ensemble Models
    "RandomForest": {
        "n_estimators": [200, 300, 500],
        "max_depth": [10, 15, 20, None],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2", 0.3],
        "class_weight": ["balanced", "balanced_subsample"],
    },
    "ExtraTrees": {
        "n_estimators": [200, 300, 500],
        "max_depth": [10, 15, 20, None],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2", 0.3],
        "class_weight": ["balanced", "balanced_subsample"],
    },
    # Linear Models
    "LogisticRegression": {
        "C": [0.01, 0.1, 1, 10],
        "class_weight": ["balanced", None],
    },
    # Neural Network
    "MLP": {
        "hidden_layer_sizes": [(64, 32), (128, 64), (128, 64, 32)],
        "alpha": [0.0001, 0.001, 0.01],
        "learning_rate_init": [0.001, 0.01],
        "batch_size": [64, 128, 256],
    },
}

In [ ]:
if PERFORM_TUNING:
    for model_name, model_data in scores[:3]:
        header(f"Hyperparameter Tuning for {model_name}")

        fit_params = {}
        if model_name == "XGBoost":
            weights = compute_sample_weight("balanced", y_train)
            fit_params["sample_weight"] = weights

        search_cv = RandomizedSearchCV(
            estimator=models[model_name],
            param_distributions=params_dict[model_name],
            n_iter=5,
            scoring="f1_macro",
            cv=3,
            verbose=3,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )

        search_cv.fit(X_train, y_train, **fit_params)
        scores.append(
            (
                f"{model_name}_Tuned",
                {
                    "score": search_cv.best_score_,
                    "model": search_cv.best_estimator_,
                },
            )
        )

        print(f"\t{model_name}_Tuned Best Score: {search_cv.best_score_:.4f}")
else:
    print("Skipping hyperparameter tuning (PERFORM_TUNING = False).")
    print("Using optimized default parameters instead.")

In [ ]:
# sort scores
scores = list(sorted(scores, key=lambda x: x[1]["score"], reverse=True))

## Ensemble Models

In [ ]:
classes = np.unique(y)
test_probs = np.zeros((X_test.shape[0], len(np.unique(y))))

n_models = min(len(scores), TOP_K)
for model_name, model_data in scores[:n_models]:
    header(f"Ensemble Model: {model_name}")
    print(f"Macro F1 Score: {model_data['score']:.4f}")

    model = model_data["model"]
    test_probs += model.predict_proba(X_test) / n_models
    classes = model.classes_

In [ ]:
header("Ensemble Model Results")
ensemble_pred_indices = np.argmax(test_probs, axis=1)
ensemble_preds = classes[ensemble_pred_indices]

ensemble_model_name = f"Ensemble_Top_{TOP_K}"
ensemble_score = f1_score(y_test, ensemble_preds, average="macro")

scores.append(
    (
        ensemble_model_name,
        {
            "score": ensemble_score,
            "model": None,
        },
    )
)

print(f"Ensemble Macro F1 Score: {ensemble_score:.4f}")

print("\nEnsemble Classification Report:")
print(classification_report(y_test, ensemble_preds))

## Model Evaluation

In [ ]:
scores = list(sorted(scores, key=lambda x: x[1]["score"], reverse=True))

header("Final Leaderboard")
print(f"{'Rank':<5} {'Model Name':<25} {'Score':<10}")
print("-" * 40)

for rank, (name, data) in enumerate(scores, 1):
    print(f"{rank:<5} {name:<25} {data['score']:.4f}")

# Update best_model_data to the new #1
best_model_name, best_model_data = scores[0]
print(
    f"\n>> New Best Model: {best_model_name} with Score: {best_model_data['score']:.4f}"
)

## Predicting Test Data

In [ ]:
if best_model_name == ensemble_model_name:
    # Use ensemble model
    final_probs = np.zeros((X_sub.shape[0], len(np.unique(y))))
    print("Generating Ensemble submission...")
    header("Retraining Top Models on Full Data")

    n_models = min(len(scores), TOP_K)
    for model_name, model_data in scores[1 : n_models + 1]:
        print(f"Retraining {model_name}...")
        print(f"Macro F1 Score: {model_data['score']:.4f}")

        print("Retraining on full training data...")
        model = model_data["model"]

        if "XGBoost" in model_name:
            sample_weights = compute_sample_weight("balanced", y)
            model.fit(X, y, sample_weight=sample_weights)
        else:
            model.fit(X, y)

        # Final predictions
        model_preds = model.predict_proba(X_sub)
        final_probs += model_preds * (1 / n_models)

        print(f"{model_name} retrained on full data...\n")
else:
    # Use single best model
    print(
        f"Ensemble score ({ensemble_score:.4f}) did not beat single best model ({best_model_data['score']:.4f})."
    )

    header(f"Best Model: {best_model_name}")
    print(f"Macro F1 Score: {best_model_data['score']:.4f}")

    print("Classification Report:")
    best_val_preds = best_model_data["model"].predict(X_test)
    print(classification_report(y_test, best_val_preds))

    print("\nRetraining on full training data...")
    model = best_model_data["model"]

    if "XGBoost" in best_model_name:
        sample_weights = compute_sample_weight("balanced", y)
        model.fit(X, y, sample_weight=sample_weights)
    else:
        model.fit(X, y)

    final_probs = model.predict_proba(X_sub)

# Final predictions
final_pred_indices = np.argmax(final_probs, axis=1)
final_preds = classes[final_pred_indices]
print("Final submission generated.")

## Final Submission

In [ ]:
submission = pd.DataFrame({"ID": test.index + 1, "label": final_preds})
submission.to_csv("submission.csv", index=False)

header("Submission created!")
print(submission["label"].value_counts())